# EEG-to-Text Model Comparison Template

**For external researchers who want to compare their model against the V9+QML baseline on ZuCo.**

## What you need before running this notebook

1. Your trained EEG-to-text model evaluated on ZuCo
2. Your TF BLEU-1 and ROUGE-1 scores (minimum)
3. The file `nat_v9_qml_results.json` from the original project (get it from the paper authors or run the product notebook yourself)
4. An NVIDIA API key (free at [build.nvidia.com](https://build.nvidia.com))

## What this notebook does

- Takes your model metrics as input
- Automatically compares them against V5, V8, V9+QML clean, and V9+QML noisy baselines
- Runs 4 domain-specific agents: Scientist, Comparator, Critic, Synthesiser
- V9+QML noisy is the hardware-realistic variant (DepolarizingChannel + PhaseDamping + MC average)
- Produces a structured JSON report you can cite

## You do NOT need to

- Install PyTorch or re-run inference
- Have access to the ZuCo dataset
- Have the original model checkpoints

Your metrics are enough.


In [1]:
# Cell 1 — Install dependencies (run once)
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import nemoguardrails
    print(f'✅ nemoguardrails {nemoguardrails.__version__}')
except ImportError:
    _pip('nemoguardrails>=0.10.0')
    print('✅ nemoguardrails installed')

try:
    import openai
    print(f'✅ openai {openai.__version__}')
except ImportError:
    _pip('openai>=1.0.0')
    print('✅ openai installed')

print('All dependencies ready.')


✅ nemoguardrails 0.21.0
✅ openai 2.30.0
All dependencies ready.


In [2]:
# Cell 2 — Setup paths and API key
import os, sys
from pathlib import Path

# ── Set your NVIDIA API key ───────────────────────────────────────────────────
os.environ['NVIDIA_API_KEY'] = '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'
os.environ.setdefault('OPENAI_API_KEY', os.environ['NVIDIA_API_KEY'])

# ── Locate this notebook's directory (works regardless of where you open it) ──
# The template notebook lives inside eeg_product/, so Path(".") IS eeg_product/
# We add it to sys.path directly — no need to descend further.
NOTEBOOK_DIR    = Path(".").resolve()          # .../project1/eeg_product
EEG_PRODUCT_DIR = NOTEBOOK_DIR                 # already the right folder
PROJECT_DIR     = NOTEBOOK_DIR.parent          # .../project1

if str(EEG_PRODUCT_DIR) not in sys.path:
    sys.path.insert(0, str(EEG_PRODUCT_DIR))

# ── Point to nat_v9_qml_results.json ─────────────────────────────────────────
# Produced by running nat_eeg_agents_v9_product.ipynb (one level up in project1/)
RESULTS_JSON = str(PROJECT_DIR / "nat_v9_qml_results.json")

# ── Verification ─────────────────────────────────────────────────────────────
key = os.environ.get("NVIDIA_API_KEY", "")
print(f"API key set    : {key.startswith('nvapi-') and len(key) > 20}")
print(f"eeg_product at : {EEG_PRODUCT_DIR}")
print(f"Results JSON   : {Path(RESULTS_JSON).exists()}  ({RESULTS_JSON})")

if not Path(RESULTS_JSON).exists():
    print()
    print("  ⚠  nat_v9_qml_results.json not found.")
    print("  Run nat_eeg_agents_v9_product.ipynb (cells 1-19) first,")
    print(f"  or copy the file to: {RESULTS_JSON}")


API key set    : True
eeg_product at : /home/deeptanshu/Desktop/project1/eeg_product
Results JSON   : True  (/home/deeptanshu/Desktop/project1/nat_v9_qml_results.json)


## Step 1 — Fill in your model details

Fill in everything you have. Only `model_name`, `tf_bleu1_pct`, and `tf_rouge1_pct` are required.
The more you fill in, the richer the agent analysis.


In [3]:
# Cell 3 — SAMPLE DATA: EEGConformer + LoRA (a realistic hypothetical submission)
# This is a realistic sample submission representing a published-style model.
# Replace all values with your own model's actual numbers before running.

from eeg_submission_schema import EEGModelSubmission, load_v9_qml_baseline, load_v9_qml_noisy_baseline

my_model = EEGModelSubmission(

    # ── Required ──────────────────────────────────────────────────────────────
    model_name        = "EEGConformer_LoRA_v1",
    architecture_desc = (
        "EEG Conformer encoder (convolutional patch embedding + 6-head self-attention, "
        "4 transformer layers) replacing GRU-based region encoders. "
        "No MoCo pretraining. LoRA rank=8 applied to GPT-2 blocks [9,10,11]. "
        "Single mean-pool across time before LoRA prefix. "
        "No SR adapter. No HTP. No QML component. "
        "Trained on ZuCo sentence-aware split, same normalisation as V8/V9."
    ),

    # Core TF metrics (%)
    tf_bleu1_pct      = 31.02,   # slightly above V8, below V9
    tf_rouge1_pct     = 36.45,

    # ── Recommended ───────────────────────────────────────────────────────────
    tf_bleu4_pct      = 4.51,
    tf_rougeL_pct     = 31.18,
    fg_bleu1_pct      = 14.87,   # free-generation BLEU-1
    bertscore_f1      = 85.71,
    tf_fg_ratio       = 2.09,    # TF/FG ratio (higher = stronger EEG conditioning)

    # Per-condition BLEU-1 (%)
    # NR = Normal Reading | TSR = Timed Silent | SR = Speed Reading
    per_condition_bleu1 = {
        "NR":  31.44,   # best condition (matches literature)
        "TSR": 33.02,   # mid
        "SR":  26.81,   # hardest — still struggles like all models
    },

    # ── Context ───────────────────────────────────────────────────────────────
    dataset           = "ZuCo",
    val_split         = "sentence-aware TEST_SIZE=0.15 seed=42",  # same as V8/V9
    n_val_samples     = 2032,
    dominant_region   = "Left_Temporal",
    eeg_channels      = 24,
    notes             = (
        "Conformer encoder replaces GRU region encoders. "
        "No quantum component. Trained 20 epochs, AdamW lr=3e-4, batch=8. "
        "Parameter count: ~117M (GPT-2 medium backbone, LoRA adds ~0.8M)."
    ),

    # ── Attention weights (optional — include if you have them) ───────────────
    # Normalised cross-region attention from final transformer layer
    # (should sum to ~1.0 across all regions)
    attention_values = {
        "Left_Temporal":     0.224,   # Wernicke's area — dominant
        "Left_Parieto_Occ":  0.187,   # VWFA — visual word form
        "Central_Parietal":  0.168,   # P300 component
        "Right_Temporal":    0.148,
        "Frontal":           0.139,
        "Occipital":         0.134,
    },

    # Per-condition attention breakdown
    attention_per_cond = {
        "NR": {
            "Left_Temporal":     0.218,
            "Left_Parieto_Occ":  0.192,
            "Central_Parietal":  0.171,
            "Right_Temporal":    0.152,
            "Frontal":           0.136,
            "Occipital":         0.131,
        },
        "TSR": {
            "Left_Temporal":     0.221,
            "Left_Parieto_Occ":  0.185,
            "Central_Parietal":  0.169,
            "Right_Temporal":    0.149,
            "Frontal":           0.141,
            "Occipital":         0.135,
        },
        "SR": {
            "Left_Temporal":     0.234,   # elevated in SR — stress response
            "Left_Parieto_Occ":  0.179,
            "Central_Parietal":  0.163,
            "Right_Temporal":    0.144,
            "Frontal":           0.142,
            "Occipital":         0.138,
        },
    },
)

# ── Preview comparison table before running agents ────────────────────────────
v9qml       = load_v9_qml_baseline(RESULTS_JSON)
v9qml_noisy = load_v9_qml_noisy_baseline(RESULTS_JSON)
my_model.print_summary(v9qml)
print(f"  V9+QML noisy : BLEU-1={v9qml_noisy.get('tf_bleu1_pct','?')}")

# ── Validation ────────────────────────────────────────────────────────────────
warnings = my_model.validate()
if warnings:
    print("⚠  Validation warnings:")
    for w in warnings:
        print(f"   - {w}")
else:
    print("✅ Submission valid — ready to run agents")


✅ V9+QML baseline loaded: BLEU-1=30.95%
✅ V9+QML noisy baseline loaded: BLEU-1=30.95%  (Δ vs clean: +0.00pp)

  Submission: EEGConformer_LoRA_v1
  Model                  BLEU-1   ROUGE-1
  ────────────────────  ────────  ────────
  V5 baseline             29.24%     33.92%
  V8 baseline             30.40%     35.78%
  V9+QML (ours)          30.95%    36.04%
  YOUR MODEL              31.02%     36.45%
    Δ vs V9+QML           +0.07pp    +0.41pp

  V9+QML noisy : BLEU-1=30.95
✅ Submission valid — ready to run agents


## Step 2 — Run the comparison pipeline

This runs 4 agents:
- **Scientist** — analyses your model on its own merits
- **Comparator** — head-to-head table vs V9+QML with per-metric verdicts
- **Critic** — challenges methodology and statistical significance
- **Synthesiser** — plain-language summary + single most important next step

Each call goes through NeMo Guardrails to ensure outputs stay within EEG research scope.


In [4]:
# Cell 4 — Run comparison pipeline
from comparison_pipeline import run_comparison_pipeline

results = await run_comparison_pipeline(
    submission       = my_model,
    results_json_path = RESULTS_JSON,
    verbose          = True,
)


✅ NeMo Guardrails loaded
✅ V9+QML baseline loaded: BLEU-1=30.95%
✅ V9+QML noisy baseline loaded: BLEU-1=30.95%  (Δ vs clean: +0.00pp)

  Submission: EEGConformer_LoRA_v1
  Model                  BLEU-1   ROUGE-1
  ────────────────────  ────────  ────────
  V5 baseline             29.24%     33.92%
  V8 baseline             30.40%     35.78%
  V9+QML (ours)          30.95%    36.04%
  YOUR MODEL              31.02%     36.45%
    Δ vs V9+QML           +0.07pp    +0.41pp

✅ Guardrails loaded (Colang 1.0) from /home/deeptanshu/Desktop/project1/eeg_product/guardrails_config
  EEG Model Comparison Pipeline
  Submitted model : EEGConformer_LoRA_v1
  Reference       : V9+QML clean + noisy (V5/V8 also in context)
  Endpoint        : https://integrate.api.nvidia.com/v1
  Rails           : NeMo Guardrails

[1/4] Scientist agent...
  ✓  latency=8313.9ms  guard=✅
[2/4] Comparator agent...
  ✓  latency=6028.3ms  guard=✅
[3/4] Critic agent...
  ✓  latency=4389.0ms  guard=✅
[4/4] Synthesiser agent...

In [5]:
# Cell 5 — Display results
from IPython.display import Markdown, display

display(Markdown('---\n## Scientist Analysis'))
display(Markdown(results['scientist']))

display(Markdown('---\n## Head-to-Head Comparison (Comparator)'))
display(Markdown(results['comparator']))

display(Markdown('---\n## Critic Review'))
display(Markdown(results['critic']))

display(Markdown('---\n## Plain-Language Synthesis'))
display(Markdown(results['synthesiser']))

# Benchmark summary
ps = results['pipeline_summary']
print(f'\n── Pipeline summary ──────────────────────────────')
print(f'  Total latency  : {ps["total_pipeline_ms"]}ms')
print(f'  Guardrail pass : {ps["guardrail_pass_rate_pct"]}%')
print(f'  Agents run     : {ps["n_agents"]}')

---
## Scientist Analysis

**Analysis of the submitted model: EEGConformer_LoRA_v1**

**1. Architecture analysis:**
The submitted model, EEGConformer_LoRA_v1, builds upon the existing EEG-to-text architecture by replacing the GRU-based region encoders with an EEG Conformer encoder. This change introduces a convolutional patch embedding layer followed by a 6-head self-attention mechanism and 4 transformer layers. Additionally, the model applies LoRA (Low-Rank Adaptation) to the GPT-2 blocks at ranks 8, which is a more aggressive adaptation compared to the baseline V9+QML model. The absence of MoCo pretraining, SR adapter, HTP, and QML components is notable.

**2. TF metrics evaluation:**
The submitted model achieves a TF BLEU-1 of 31.02%, which is an improvement over the baseline V8 model (30.40%) but slightly lower than the V9+QML clean variant (30.95%). The TF ROUGE-1 metric is 36.45%, which is also higher than V8 (35.78%) but comparable to V9+QML clean (36.04%). The BERTScore of 85.71% is slightly lower than V9+QML clean (85.46%) but still indicates a strong performance.

**3. Per-condition performance assessment:**
The submitted model shows a slight improvement in per-condition performance, with NR (31.44) and TSR (33.02) scores higher than V9+QML clean (31.43 and 33.87, respectively). However, the SR score (26.81) is lower than V9+QML clean (27.37). This suggests that the model may not handle speed reading conditions as well as the baseline V9+QML model.

**4. TF/FG ratio analysis:**
The TF/FG ratio of 2.09 indicates that the model's EEG conditioning strength is relatively weak compared to the baseline V9+QML model (4.5×). This may be due to the absence of QML components and the more aggressive LoRA adaptation.

**5. Missing metrics:**
The submitted model lacks metrics such as val_loss, which would provide insight into the model's generalization performance. Additionally, the absence of metrics like accuracy or F1-score for specific conditions (e.g., NR, TSR, SR) would be beneficial for a more comprehensive evaluation.

**6. Strengths and weaknesses:**

Strengths:

1. **Improved TF metrics:** The submitted model achieves higher TF BLEU-1 and ROUGE-1 scores compared to the baseline V8 model.
2. **Strong BERTScore:** The model's BERTScore of 85.71% indicates a strong performance in terms of semantic similarity.
3. **Efficient architecture:** The use of LoRA adaptation and the absence of QML components may result in a more efficient architecture, which is beneficial for deployment.

Weaknesses:

1. **Lower SR performance:** The model's SR score (26.81) is lower than the baseline V9+QML clean variant (27.37), indicating a potential weakness in handling speed reading conditions.
2. **Weak EEG conditioning strength:** The TF/FG ratio of 2.09 suggests that the model's EEG conditioning strength is relatively weak compared to the baseline V9+QML model.

---
## Head-to-Head Comparison (Comparator)

**Head-to-Head Comparison Report**

**EEGConformer_LoRA_v1 vs V9+QML clean**

| METRIC NAME | Submitted: X%    V9+QML: Y%    Delta: ±Z pp    Verdict: BETTER / WORSE / EQUIVALENT / N/A |
| --- | --- | --- | --- | --- |
| TF BLEU-1 | 31.02 | 30.95 | ±0.07 | EQUIVALENT |
| TF BLEU-4 | 4.51 | 4.46 | ±0.05 | EQUIVALENT |
| TF ROUGE-1 | 36.45 | 36.04 | ±0.41 | BETTER |
| TF ROUGE-L | 31.18 | 30.77 | ±0.41 | BETTER |
| FG BLEU-1 | 14.87 | 6.88 | ±7.99 | WORSE |
| BERTScore F1 | 85.71 | null | N/A | N/A |
| TF FG Ratio | 2.09 | 4.5 | -2.41 | WORSE |
| Per-Condition NR | 31.44 | 31.43 | ±0.01 | EQUIVALENT |
| Per-Condition TSR | 33.02 | 33.87 | -0.85 | WORSE |
| Per-Condition SR | 26.81 | 27.37 | -0.56 | WORSE |

**EEGConformer_LoRA_v1 vs V9+QML noisy**

| METRIC NAME | Submitted: X%    V9+QML: Y%    Delta: ±Z pp    Verdict: BETTER / WORSE / EQUIVALENT / N/A |
| --- | --- | --- | --- | --- |
| TF BLEU-1 | null | null | N/A | N/A |
| TF ROUGE-1 | null | null | N/A | N/A |

**OVERALL VERDICT vs CLEAN:**
BELOW

**OVERALL VERDICT vs NOISY:**
INSUFFICIENT DATA

**STRONGEST IMPROVEMENT:**
The submitted model shows a significant improvement in TF ROUGE-1 (±0.41) and TF ROUGE-L (±0.41) compared to the baseline V9+QML clean model.

**BIGGEST GAP:**
The submitted model shows a significant gap in FG BLEU-1 (±7.99) compared to the baseline V9+QML clean model.

**NOISE ROBUSTNESS NOTE:**
The submitted model does not have any noise-awareness components, as it is trained on a noiseless statevector simulation.

**FAIRNESS NOTE:**
The comparison is apple-to-apple, with the same split (sentence-aware) and evaluation metrics.

---
## Critic Review

**ISSUE-1: Eval protocol**
Problem: The submitted model uses a different validation split (TEST_SIZE=0.15 seed=42) than the standard protocol, which may affect the comparison with V9+QML.
Fix: Verify that the submitted model uses the same validation split as the standard protocol.

**ISSUE-2: Metric completeness**
Problem: The submitted model does not provide BERTScore F1 for the noisy condition, and TF FG BLEU is missing for the clean condition.
Fix: Provide BERTScore F1 for the noisy condition and TF FG BLEU for the clean condition.

**ISSUE-3: Architecture comparison fairness**
Problem: The submitted model has more parameters (~117M) than V9+QML, which may give it an unfair advantage.
Fix: Compare the performance of the submitted model with a smaller architecture or a similar architecture with fewer parameters.

**ISSUE-4: Per-condition analysis**
Problem: The submitted model performs worse on SR condition, which is the most challenging condition.
Fix: Investigate the reasons for the poor performance on SR condition and provide a more detailed analysis.

**ISSUE-5: Statistical significance**
Problem: The submitted model shows small deltas (~0.05-0.07 pp) on some metrics, which may not be statistically significant.
Fix: Perform a statistical significance test to determine if the observed deltas are significant.

**ISSUE-6: Noise robustness**
Problem: The submitted model does not show any noise-awareness, as its performance degrades significantly in the noisy condition.
Fix: Investigate the reasons for the poor performance in the noisy condition and provide a more detailed analysis.

**Correctly identified contributions:**
* Replacing GRU region encoders with an EEG Conformer encoder
* Applying LoRA to the GPT-2 blocks
* Providing attention weights for different conditions

**Verdict: CONDITIONAL ACCEPT**
The submitted model shows some improvements over V9+QML, but the issues mentioned above need to be addressed before acceptance.

**Confidence: 6/10**
The submission shows some promising results, but the lack of fairness in architecture comparison and the poor performance on SR condition are significant concerns.

---
## Plain-Language Synthesis

The submitted model, EEGConformer_LoRA_v1, is an EEG-to-text architecture that builds upon the existing V9+QML model by introducing an EEG Conformer encoder and applying LoRA to the GPT-2 blocks at ranks 8. This change aims to improve the model's performance on EEG-based text generation tasks. The model's architecture is designed to take advantage of the convolutional patch embedding layer and the 6-head self-attention mechanism, which are expected to improve the model's ability to capture complex patterns in the EEG data.

In a head-to-head comparison with the V9+QML model, EEGConformer_LoRA_v1 shows mixed results. On the clean condition, the model performs similarly to V9+QML on TF BLEU-1 and TF BLEU-4, but outperforms it on TF ROUGE-1 and TF ROUGE-L. However, on the noisy condition, the model's performance is significantly worse than V9+QML on FG BLEU-1 and TF FG Ratio. The clean vs noisy gap in val loss is 0.0004, which translates to a small difference in BLEU score (~0pp). This suggests that the model may be more sensitive to noise in the EEG data.

The critic's concerns highlight some potential issues with the evaluation protocol and metric completeness. Specifically, the submitted model uses a different validation split than the standard protocol, which may affect the comparison with V9+QML. Additionally, the model does not provide BERTScore F1 for the noisy condition, and TF FG BLEU is missing for the clean condition. These issues need to be addressed to ensure a fair comparison between the two models.

In terms of practical implications, the critic's concerns suggest that the comparison between EEGConformer_LoRA_v1 and V9+QML may not be entirely reliable. This means that researchers should approach the results with caution and consider the potential biases in the evaluation protocol. To strengthen the model or the evaluation, the submitting researcher should verify that the model uses the same validation split as the standard protocol, provide BERTScore F1 for the noisy condition, and ensure that all relevant metrics are reported.

The single most important next step for EEGConformer_LoRA_v1 is to retrain the model using the standard validation split and report all relevant metrics, including BERTScore F1 for the noisy condition.


── Pipeline summary ──────────────────────────────
  Total latency  : 23475.7ms
  Guardrail pass : 100.0%
  Agents run     : 4


In [6]:
# Cell 6 — Save report
from comparison_pipeline import save_comparison_report

output_file = save_comparison_report(results)
print(f'Report saved to: {output_file}')
print('You can share this JSON file with collaborators or cite it in your paper.')


✅ Comparison report saved → comparison_eegconformer_lora_v1.json
Report saved to: comparison_eegconformer_lora_v1.json
You can share this JSON file with collaborators or cite it in your paper.


---
## Understanding the output

### Comparator verdicts

| Verdict | Meaning |
|---------|--------|
| BETTER | Your model beats V9+QML by > 0.5pp |
| EQUIVALENT | Within ±0.5pp (not meaningfully different) |
| WORSE | Your model is below V9+QML by > 0.5pp |
| N/A | Metric not provided in your submission |

### Per-condition interpretation

The three ZuCo conditions test different cognitive states:
- **NR (Normal Reading)**: Standard reading, easiest for decoding
- **TSR (Timed Silent Read)**: Time-pressured silent reading, moderate
- **SR (Speed Reading)**: Maximum speed, hardest for the decoder

Improvement on SR is the most meaningful — all models struggle there.

### V9+QML clean vs noisy

Both QML variants are provided as reference baselines:
- **V9+QML clean** — noiseless statevector simulation (lightning.qubit). Best val loss: 4.1733
- **V9+QML noisy** — hardware-realistic simulation (DepolarizingChannel p=0.01 + PhaseDamping γ=0.02). Best val loss: 4.1729

The 0.0004 val loss gap shows the architecture is hardware-deployable — real quantum noise at this scale causes negligible degradation. When comparing your model, beating either variant is meaningful. Beating V9+QML noisy means your model outperforms the hardware-ready variant.

### What the TF/FG ratio means

- **< 2×**: EEG prefix strongly conditions the LM
- **2–3×**: Partial conditioning (where most models sit)
- **> 3×**: LM largely ignoring the EEG signal

V8 was 1.97×. V9+QML clean and noisy are reported in the comparison. Where does yours sit?

---

## Questions and contact

If you have questions about the baseline models, the ZuCo evaluation protocol, or the V5→V8→V9→QML architecture chain, refer to:
- The `nat_eeg_agents_v9_product.ipynb` notebook
- The `eeg_submission_schema.py` file for the full schema documentation
  (includes `V9_QML_NOISY_BASELINE` and `load_v9_qml_noisy_baseline()`)
- The `comparison_pipeline.py` file for agent prompt details
- `hybrid_qml_noisy_v9_best.pt` — the noise-trained checkpoint (request from authors)
